In [1]:
!pip install flask requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import subprocess, time, requests

In [3]:
%%file app.py
from flask import Flask

app = Flask(__name__)

@app.route("/")                      # URL: http://localhost:5000/
def home():
    return "Welcome to the transaction monitoring system!"

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [4]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

In [5]:
response = requests.get("http://localhost:5000/")
print(f"Status: {response.status_code}")
print(f"Body: {response.text}")

Status: 200
Body: Welcome to the transaction monitoring system!


In [6]:
server.kill()

In [7]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/")
def home():
    return "Welcome to the transaction monitoring system!"

@app.route("/hello")
def hello():
    name = request.args.get("name", "stranger")
    return f"Hello, {name}!"

@app.route("/transaction/<tx_id>")
def get_transaction(tx_id):
    return jsonify({
        "tx_id": tx_id,
        "status": "found"
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [8]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

In [9]:
r1 = requests.get("http://localhost:5000/hello")
r2 = requests.get("http://localhost:5000/hello?name=Anna")

print(r1.text)
print(r2.text)

Hello, stranger!
Hello, Anna!


In [10]:
r3 = requests.get("http://localhost:5000/transaction/TX0042")
print(r3.json())

{'status': 'found', 'tx_id': 'TX0042'}


In [11]:
server.kill()

In [12]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/")
def home():
    return "Welcome to the transaction monitoring system!"

@app.route("/hello")
def hello():
    name = request.args.get("name", "stranger")
    return f"Hello, {name}!"

@app.route("/transaction/<tx_id>")
def get_transaction(tx_id):
    return jsonify({
        "tx_id": tx_id,
        "status": "found"
    })

@app.route("/status")
def status():
    return jsonify({
        "service": "monitoring",
        "version": "1.0",
        "status": "ok"
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [13]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

In [14]:
r = requests.get("http://localhost:5000/status")
print(r.status_code)
print(r.json())

200
{'service': 'monitoring', 'status': 'ok', 'version': '1.0'}


In [15]:
server.kill()

In [16]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/echo", methods=["POST"])
def echo():
    data = request.get_json()
    return jsonify({
        "received": data,
        "field_count": len(data)
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [17]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

In [18]:
transaction = {
    "tx_id": "TX0042",
    "amount": 4500.0,
    "store": "Krakow",
    "category": "electronics"
}

r = requests.post("http://localhost:5000/echo", json=transaction)
print(r.json())

{'field_count': 4, 'received': {'amount': 4500.0, 'category': 'electronics', 'store': 'Krakow', 'tx_id': 'TX0042'}}


In [19]:
r_bad = requests.get("http://localhost:5000/echo")
print(f"Status: {r_bad.status_code}")

Status: 405


In [ ]:
# ANSWER:
# Status code 405 means Method Not Allowed.
# The endpoint exists, but it only accepts POST, not GET.

In [20]:
server.kill()

In [21]:
def score_transaction(tx: dict) -> dict:
    """
    Assess transaction risk using business rules.
    Returns dict with: score, risk_level, triggered_rules.
    """
    score = 0
    rules = []

    if tx.get("amount", 0) > 3000:
        score += 3
        rules.append("R1: amount > 3000")

    if tx.get("category") == "electronics" and tx.get("amount", 0) > 1500:
        score += 2
        rules.append("R2: electronics > 1500")

    if tx.get("hour", 12) < 6:
        score += 2
        rules.append("R3: night hour")

    if score >= 5:
        risk_level = "CRITICAL"
    elif score >= 3:
        risk_level = "HIGH"
    elif score >= 1:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    return {
        "score": score,
        "risk_level": risk_level,
        "triggered_rules": rules
    }

In [22]:
test_tx = {
    "tx_id": "TX001",
    "amount": 4500.0,
    "category": "electronics",
    "hour": 3
}

print(score_transaction(test_tx))

{'score': 7, 'risk_level': 'CRITICAL', 'triggered_rules': ['R1: amount > 3000', 'R2: electronics > 1500', 'R3: night hour']}


In [23]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

def score_transaction(tx):
    score = 0
    rules = []

    if tx.get("amount", 0) > 3000:
        score += 3
        rules.append("R1: amount > 3000")

    if tx.get("category") == "electronics" and tx.get("amount", 0) > 1500:
        score += 2
        rules.append("R2: electronics > 1500")

    if tx.get("hour", 12) < 6:
        score += 2
        rules.append("R3: night hour")

    if score >= 5:
        risk_level = "CRITICAL"
    elif score >= 3:
        risk_level = "HIGH"
    elif score >= 1:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    return {
        "score": score,
        "risk_level": risk_level,
        "triggered_rules": rules
    }

@app.route("/score", methods=["POST"])
def score():
    tx = request.get_json()

    if not tx or "amount" not in tx:
        return jsonify({
            "error": "Missing required field 'amount'"
        }), 400

    result = score_transaction(tx)
    result["tx_id"] = tx.get("tx_id", "unknown")

    return jsonify(result)

@app.route("/health")
def health():
    return jsonify({
        "status": "ok",
        "version": "1.0-rules"
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [24]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

In [27]:
# Step 4.2.1 — Test /score with 3 transactions
cases = [
    {
        "tx_id": "TX001",
        "amount": 50.0,
        "category": "food",
        "hour": 14
    },
    {
        "tx_id": "TX002",
        "amount": 1800.0,
        "category": "electronics",
        "hour": 10
    },
    {
        "tx_id": "TX003",
        "amount": 4500.0,
        "category": "electronics",
        "hour": 3
    }
]

for tx in cases:
    r = requests.post("http://localhost:5000/score", json=tx)
    res = r.json()

    print(
        f"{tx['tx_id']} "
        f"{tx['amount']:>7.0f} PLN -> "
        f"{res['risk_level']:8s} "
        f"(score={res['score']}) "
        f"{res['triggered_rules']}"
    )

TX001      50 PLN -> LOW      (score=0) []
TX002    1800 PLN -> MEDIUM   (score=2) ['R2: electronics > 1500']
TX003    4500 PLN -> CRITICAL (score=7) ['R1: amount > 3000', 'R2: electronics > 1500', 'R3: night hour']


In [28]:
# Task 4.3 — Test error handling
r = requests.post(
    "http://localhost:5000/score",
    json={
        "tx_id": "TX000",
        "category": "food",
        "hour": 12
    }
)

print(r.status_code)
print(r.json())

400
{'error': "Missing required field 'amount'"}


In [29]:
# Task 4.4 — Review questions

# Question 1 — Difference between GET and POST
# ANSWER:
# GET is used to request or fetch data from the server.
# POST is used to send data to the server, usually in the request body.
# In this lab, /hello uses GET, while /score uses POST because it sends transaction JSON.

# Question 2 — Why use jsonify()?
# ANSWER:
# jsonify() converts a Python dictionary into a proper JSON response.
# It also sets the correct response header: Content-Type: application/json.
# This makes it easier for clients to read the response as JSON.

# Question 3 — What happens if two people call /score at the same time?
# ANSWER:
# Flask can handle multiple requests, but the basic development server is not designed for production.
# If two people call /score at the same time, each request is processed separately.
# Since our scoring function does not modify shared data, the result should be safe.
# In production, we would use a proper server like Gunicorn.

In [35]:
# Homework Step 1 — Stop current server
server.kill()

### Homework Step 2 — Final app.py with /score, /health, and /stats

In [38]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

counters = {
    "total": 0,
    "high": 0,
    "critical": 0
}

def score_transaction(tx):
    score = 0
    rules = []

    if tx.get("amount", 0) > 3000:
        score += 3
        rules.append("R1: amount > 3000")

    if tx.get("category") == "electronics" and tx.get("amount", 0) > 1500:
        score += 2
        rules.append("R2: electronics > 1500")

    if tx.get("hour", 12) < 6:
        score += 2
        rules.append("R3: night hour")

    if score >= 5:
        risk_level = "CRITICAL"
    elif score >= 3:
        risk_level = "HIGH"
    elif score >= 1:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    return {
        "score": score,
        "risk_level": risk_level,
        "triggered_rules": rules
    }

@app.route("/score", methods=["POST"])
def score():
    tx = request.get_json()

    if not tx or "amount" not in tx:
        return jsonify({
            "error": "Missing required field 'amount'"
        }), 400

    if tx["amount"] < 0:
        return jsonify({
            "error": "Amount cannot be negative"
        }), 400

    result = score_transaction(tx)
    result["tx_id"] = tx.get("tx_id", "unknown")

    counters["total"] += 1

    if result["risk_level"] == "HIGH":
        counters["high"] += 1

    if result["risk_level"] == "CRITICAL":
        counters["critical"] += 1

    return jsonify(result)

@app.route("/stats")
def stats():
    return jsonify(counters)

@app.route("/health")
def health():
    return jsonify({
        "status": "ok",
        "version": "1.0-rules"
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


### Homework Step 3 — Start server

In [39]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

### Homework Step 4 — Test negative amount validation

In [40]:
bad_tx = {
    "tx_id": "TX_NEG",
    "amount": -100,
    "category": "food",
    "hour": 12
}

r = requests.post("http://localhost:5000/score", json=bad_tx)

print(r.status_code)
print(r.json())

400
{'error': 'Amount cannot be negative'}


In [41]:
transactions = [
    {"tx_id": "TX001", "amount": 50, "category": "food", "hour": 14},
    {"tx_id": "TX002", "amount": 1800, "category": "electronics", "hour": 10},
    {"tx_id": "TX003", "amount": 4500, "category": "electronics", "hour": 3},
    {"tx_id": "TX004", "amount": 3200, "category": "clothing", "hour": 15},
    {"tx_id": "TX005", "amount": 200, "category": "food", "hour": 2},
    {"tx_id": "TX006", "amount": 1600, "category": "electronics", "hour": 1},
    {"tx_id": "TX007", "amount": 700, "category": "books", "hour": 11},
    {"tx_id": "TX008", "amount": 5000, "category": "travel", "hour": 23},
    {"tx_id": "TX009", "amount": 2500, "category": "electronics", "hour": 5},
    {"tx_id": "TX010", "amount": 100, "category": "food", "hour": 9}
]

results = []

for tx in transactions:
    r = requests.post("http://localhost:5000/score", json=tx)
    res = r.json()

    results.append({
        "tx_id": res["tx_id"],
        "amount": tx["amount"],
        "category": tx["category"],
        "hour": tx["hour"],
        "score": res["score"],
        "risk_level": res["risk_level"],
        "rules": res["triggered_rules"]
    })

### Homework Step 6 — Print summary table

In [42]:
print(f"{'TX ID':<8} {'Amount':>8} {'Category':<15} {'Hour':>5} {'Score':>6} {'Risk':<10} Rules")
print("-" * 90)

for row in results:
    print(
        f"{row['tx_id']:<8} "
        f"{row['amount']:>8.0f} "
        f"{row['category']:<15} "
        f"{row['hour']:>5} "
        f"{row['score']:>6} "
        f"{row['risk_level']:<10} "
        f"{row['rules']}"
    )

TX ID      Amount Category         Hour  Score Risk       Rules
------------------------------------------------------------------------------------------
TX001          50 food               14      0 LOW        []
TX002        1800 electronics        10      2 MEDIUM     ['R2: electronics > 1500']
TX003        4500 electronics         3      7 CRITICAL   ['R1: amount > 3000', 'R2: electronics > 1500', 'R3: night hour']
TX004        3200 clothing           15      3 HIGH       ['R1: amount > 3000']
TX005         200 food                2      2 MEDIUM     ['R3: night hour']
TX006        1600 electronics         1      4 HIGH       ['R2: electronics > 1500', 'R3: night hour']
TX007         700 books              11      0 LOW        []
TX008        5000 travel             23      3 HIGH       ['R1: amount > 3000']
TX009        2500 electronics         5      4 HIGH       ['R2: electronics > 1500', 'R3: night hour']
TX010         100 food                9      0 LOW        []


### Homework Step 7 — Check /stats

In [43]:
r = requests.get("http://localhost:5000/stats")
print(r.status_code)
print(r.json())

200
{'critical': 1, 'high': 4, 'total': 10}


### Final cleanup — Stop server

In [44]:
server.kill()
print("Server stopped.")

Server stopped.
